In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
!gcloud config set project YOUR_GCP_PROJECT_ID
%cd /content/drive/MyDrive/E231project
!ls centralized infra scripts config.yaml
!grep -c "driver_memory\|model_num_clients" config.yaml   # 2 = new settings present
!grep -c "_save_federated_model" federated/fedavg_coordinator.py   # > 0 = federated model saving

In [ ]:
# The canonical training data must exist (built earlier by the ETL build)
!gcloud storage ls gs://YOUR_BUCKET/full/processed/

In [ ]:
!CONFIG=config.yaml bash infra/dataproc_launch.sh 16

In [ ]:
# Final (canonical) centralized model on the FULL training set -> model bundle for Kafka scoring
#    Saved to gs://YOUR_BUCKET/full/results/models/centralized/ (als/, user_bias, item_bias, meta.json)
#    plus centralized_metrics.json and coldstart_metrics.json
import yaml, shutil, os
cfg = yaml.safe_load(open("config.yaml")); I = cfg["infra"]
stage = "/tmp/stage"; shutil.rmtree(stage, ignore_errors=True)
for pkg in ["etl", "centralized", "evaluation", "federated", "benchmarks"]:
    shutil.copytree(pkg, f"{stage}/{pkg}", ignore=shutil.ignore_patterns("__pycache__"))
shutil.make_archive("/tmp/code", "zip", stage)
CFG = cfg["data"]["results_prefix"].rstrip("/").rsplit("/", 1)[0] + "/config.yaml"   # gs://YOUR_BUCKET/full/config.yaml
!gcloud storage cp config.yaml {CFG}
n = 16 * int(I["executors_per_node"])
P = (f"spark.dynamicAllocation.enabled=false,spark.executor.instances={n},spark.executor.cores={I['executor_cores']},"
     f"spark.executor.memory={I['executor_memory']},spark.executor.memoryOverhead={I['executor_memory_overhead']},"
     f"spark.driver.memory={I['driver_memory']},spark.driver.maxResultSize={I['driver_max_result_size']}")
!gcloud dataproc jobs submit pyspark scripts/run_centralized.py --cluster={I['cluster_name']} --region={I['region']} --py-files=/tmp/code.zip --properties={P} -- --config={CFG} 2>&1 | grep -E "^\[centralized|Traceback|Error|state:"

In [ ]:
# Final federated model on the FULL training set (client count = federated.model_num_clients)
#    Saved to gs://YOUR_BUCKET/full/results/models/federated/ (user/item factors, biases, meta.json)
#    plus federated_convergence.csv and comparison.json (vs the centralized model above).
#    Only the model's client count is run here; for the full 1/4/8/16 comparison, drop --num-clients.
M = cfg["federated"]["model_num_clients"]
!gcloud dataproc jobs submit pyspark scripts/run_federated.py --cluster={I['cluster_name']} --region={I['region']} --py-files=/tmp/code.zip --properties={P} -- --config={CFG} --num-clients {M} 2>&1 | grep -E "^\[federated|Traceback|Error|state:"

In [ ]:
# Both saved model bundles (the Kafka stream scorer loads one, chosen by streaming.model)
!gcloud storage ls gs://YOUR_BUCKET/full/results/models/centralized/ gs://YOUR_BUCKET/full/results/models/federated/
!gcloud storage cat gs://YOUR_BUCKET/full/results/models/federated/meta.json
!gcloud storage cat gs://YOUR_BUCKET/full/results/comparison.json

In [ ]:
!CONFIG=config.yaml bash infra/dataproc_delete.sh